In [4]:
import pandas as pd
import numpy as np

from utils.config import load_config
from preprocessing.house_prices_preprocessing import HousePricesPreprocessor
from models.baseline.regression import run_baseline_regression

In [5]:
config = load_config()

data = pd.read_csv(config["paths"]["train"])

# --- разделяем target ---
target = "SalePrice"

# логарифмируем таргет
y = np.log1p(data[target])

X = data.drop(columns=[target])

# --- preprocessing ---
preprocessor = HousePricesPreprocessor(outlier_quantile=0.95)

X_prepared = preprocessor.fit_transform(X)

X_prepared.head()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,GarageFinish_Unf,GarageFinish_Fin,GarageFinish_None,BsmtFinType1_GLQ,BsmtFinType1_ALQ,BsmtFinType1_Unf,BsmtFinType1_Rec,BsmtFinType1_BLQ,BsmtFinType1_None,BsmtFinType1_LwQ
0,0.073375,-0.208804,-0.332210,0.651479,-0.517200,1.050994,0.878668,0.739648,0.667140,-0.327561,...,0,0,0,1,0,0,0,0,0,0
1,-0.872563,0.646406,-0.009592,-0.071836,2.179628,0.156734,-0.429577,-0.654947,1.327216,-0.327561,...,0,0,0,0,1,0,0,0,0,0
2,0.073375,-0.037762,0.453294,0.651479,-0.517200,0.984752,0.830215,0.497729,0.133255,-0.327561,...,0,0,0,1,0,0,0,0,0,0
3,0.309859,-0.493874,-0.023619,0.651479,-0.517200,-1.863632,-0.720298,-0.654947,-0.521967,-0.327561,...,1,0,0,0,1,0,0,0,0,0
4,0.073375,0.874462,1.297710,1.374795,-0.517200,0.951632,0.733308,1.835402,0.543376,-0.327561,...,0,0,0,1,0,0,0,0,0,0


In [6]:
results = run_baseline_regression(X_prepared, y)
results

,model,MAE,MSE,RMSE,R2
0,Ridge,0.088157,0.016885,0.129944,0.909515
1,LinearRegression,0.090305,0.017227,0.131253,0.907683
2,Lasso,0.162267,0.053968,0.232310,0.710799
3,Dummy,0.334953,0.186869,0.432283,-0.001381


# Baseline модели: результаты и выводы

## Задача

Предсказать цену дома (`SalePrice`) на основе признаков.

---

## Результаты baseline моделей (log target)

| Model            | MAE     | RMSE     | R²        |
| ---------------- |---------|----------|-----------|
| Ridge            | 0.0882  | **0.1299** | **0.9095** |
| LinearRegression | 0.0903  | 0.1313   | 0.9077    |
| Lasso            | 0.1623  | 0.2323   | 0.7108    |
| Dummy            | 0.3350  | 0.4323   | ~0        |

---

## Интерпретация

### 1. Что означают метрики после логарифмирования

Ты используешь:

y = log(1 + SalePrice)

Это значит:

- модель обучается не на цене, а на её логарифме
- ошибки считаются в лог-пространстве

Поэтому значения вроде RMSE = 0.129 — это нормально

---

### 2. Как интерпретировать RMSE в логах

RMSE ≈ 0.13 означает:

exp(0.13) ≈ 1.14

Ошибка модели ≈ 14%

То есть:

- если дом стоит 200 000
- ошибка ≈ ±28 000

---

### 3. Сравнение с Dummy

- Dummy: RMSE ≈ 0.43
- Ridge: RMSE ≈ 0.13

Модель уменьшает ошибку более чем в 3 раза

---

### 4. R² (коэффициент детерминации)

R² = 0.91

Модель объясняет ~91% вариации лог-цены

Это:

- выше, чем было без логарифма
- означает более стабильную модель

---

## Выводы

- Логарифмирование таргета сильно улучшило качество
- Ridge показывает лучший результат
- Линейные модели уже дают очень хороший baseline
- Dummy значительно хуже → модель извлекает полезный сигнал
- Ошибка модели ≈ 10–15%, что является хорошим результатом